# 06 — Diffusion models

**Mathematical Foundations of Control and Machine Learning — Winter 2026**

A diffusion model generates data by **running a stochastic differential equation backwards in time**. Almost every ingredient is classical: the heat equation (Fourier, 1822), Brownian motion (Einstein 1905, Wiener 1923), the Feynman–Kac formula (Kac, 1949), and the time reversal of diffusions (Anderson, 1982). The one genuinely new ingredient is that the single object the reversal needs — the **score** $\nabla \log p_t$ — can be learned from samples alone (Hyvärinen 2005, Vincent 2011, Song–Ermon 2019, Ho et al. 2020).

1. **Noising is the heat equation** — destroying data as a PDE
2. **Feynman–Kac** — the dictionary between PDEs and stochastic processes
3. **The forward process** — an Ornstein–Uhlenbeck bridge to pure noise
4. **Reversing time** — Anderson's SDE and the probability-flow ODE
5. **Score matching** — regressing on noise to learn $\nabla \log p_t$
6. **An analytically solvable example** — the lemniscate of Bernoulli
7. **A small diffusion model on MNIST** — and an animation of the denoising
8. **Outlook: diffusion as stochastic control**, and references

In [ ]:
%matplotlib inline
import gzip, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from plotting import *            # the course palette, the house style, and the figure helpers

# Reproducibility
rng = np.random.default_rng(0)
torch.manual_seed(0)

print("PyTorch version:", torch.__version__)

## 1. Noising is the heat equation

Take a data point $X_0 \sim p_{\text{data}}$ in $\mathbb{R}^d$ and let it be carried away by Brownian motion,

$$X_t = X_0 + W_t.$$

Because $W_t \sim \mathcal{N}(0, tI)$ is independent of $X_0$, the density of $X_t$ is a **convolution**

$$p_t(x) = \int k_t(x-y)\, p_{\text{data}}(y)\, dy, \qquad k_t(z) = (2\pi t)^{-d/2} e^{-|z|^2/(2t)} .$$

The Gaussian kernel satisfies $\partial_t k_t = \tfrac12 \Delta k_t$, and differentiating under the integral sign gives

$$\boxed{\;\partial_t p_t = \tfrac{1}{2}\Delta p_t, \qquad p_0 = p_{\text{data}}.\;}$$

**Adding noise to data is the heat equation run forward on the data density.** In Fourier variables $\hat p_t(\xi) = \hat p_0(\xi)\, e^{-t|\xi|^2/2}$: fine detail is destroyed first, exponentially fast in $|\xi|^2$.

Generation is the *inverse* problem: from $p_T \approx$ noise, recover $p_0$. The backward heat equation is the textbook example of an **ill-posed** problem — the amplification factor $e^{+t|\xi|^2/2}$ turns any perturbation into an explosion. The trick of diffusion models is *not* to reverse the PDE. It is to reverse the **stochastic process**, which, as we will see, is perfectly well posed provided we know one extra object: $\nabla \log p_t$.

## 2. Feynman–Kac: the PDE / stochastic process dictionary

The bridge between the two worlds has two directions.

**Forward (Kolmogorov forward / Fokker–Planck).** If $X_t$ solves $dX_t = f(X_t,t)\,dt + g(t)\,dW_t$, then its density evolves by

$$\partial_t p_t = -\nabla\!\cdot\!\big(f\,p_t\big) + \tfrac{g(t)^2}{2}\,\Delta p_t .$$

This is a statement about **densities**: it pushes the law of the particles forward.

**Backward (Feynman–Kac).** Fix a terminal payoff $\varphi$ and a potential $V$. Then

$$u(x,t) = \mathbb{E}\left[\, e^{-\int_t^T V(X_s)\,ds}\,\varphi(X_T) \;\middle|\; X_t = x \right]$$

solves the **linear** PDE

$$\partial_t u + f\cdot\nabla u + \tfrac{g^2}{2}\Delta u = V u, \qquad u(\cdot, T) = \varphi .$$

This is a statement about **observables**: the value of the PDE at $(x,t)$ is an average over all trajectories started at $x$.

Taking $f = 0$, $g = 1$, $V = 0$ and reversing time ($t \mapsto T-t$) gives the cleanest instance of the formula, the one we care about:

$$\partial_t u = \tfrac12 \Delta u, \quad u(\cdot,0) = u_0 \qquad \Longleftrightarrow \qquad u(x,t) = \mathbb{E}\big[\,u_0(x + W_t)\,\big].$$

*Solve a PDE* and *average over random paths* are the same computation. The next cell checks this numerically three ways: exact Gaussian convolution, an explicit finite-difference scheme, and a Monte Carlo average over simulated Brownian paths.

In [ ]:
# The heat equation, solved three ways at t = 0.3
x = np.linspace(-4, 4, 401)
dx = x[1] - x[0]
u0 = lambda z: np.exp(-(z + 1.5) ** 2 / 0.05) + 1.0 * ((z > 0.5) & (z < 2.0))   # bump + step
t_end = 0.3

# (a) exact: convolution with the heat kernel
K = np.exp(-(x[:, None] - x[None, :]) ** 2 / (2 * t_end)) / np.sqrt(2 * np.pi * t_end)
u_exact = K @ u0(x) * dx

# (b) explicit finite differences (stable while dt <= dx^2)
u_fd, dt = u0(x).copy(), 0.4 * dx ** 2
for _ in range(int(t_end / dt)):
    u_fd[1:-1] += 0.5 * dt / dx ** 2 * (u_fd[2:] - 2 * u_fd[1:-1] + u_fd[:-2])

# (c) Feynman-Kac: simulate Brownian paths and average the initial datum at their endpoints
M, n_steps = 8000, 200
paths = np.sqrt(t_end / n_steps) * rng.standard_normal((M, n_steps))
paths = np.cumsum(paths, axis=1)
u_mc = u0(x[:, None] + paths[None, :, -1]).mean(axis=1)

fig, (axL, axR) = panels(2, size=(4.75, 3.6))
axL.plot(np.linspace(0, t_end, n_steps), paths[:40].T, lw=0.7, alpha=0.7, color=BLUE)
label(axL, "$t$", "$W_t$", "Brownian paths started at $x$")
curves(axR, x, {"$u_0$": (u0(x), {"color": GREY, "lw": 1.2}),
                "heat kernel (exact)": (u_exact, {"color": BLUE, "lw": 2.5}),
                "finite differences": (u_fd, {"color": ORANGE, "ls": "--", "lw": 1.8}),
                r"Monte Carlo $\mathbb{E}[u_0(x+W_t)]$": (u_mc, {"color": AQUA, "ls": ":", "lw": 2.0})},
       xlabel="$x$", title=f"$u(\\cdot, t)$ at $t = {t_end}$")
plt.show()

print(f"max |FD - exact|          = {np.abs(u_fd - u_exact).max():.2e}")
print(f"max |Monte Carlo - exact| = {np.abs(u_mc - u_exact).max():.2e}   (M = {M} paths)")

The finite-difference solution agrees to discretization error; the Monte Carlo estimate agrees to $\mathcal{O}(M^{-1/2})$ statistical noise. Same object, two computational routes.

> **Remark (why this matters for a control course).** The Feynman–Kac formula with a potential is the *linear* face of a nonlinear control problem. If $u = e^{-V/\lambda}$ — the Cole–Hopf / Hopf–Cole logarithmic transform — the linear PDE above becomes a Hamilton–Jacobi–Bellman equation, and the expectation becomes a stochastic optimal control problem. We return to this in §8: sampling from a diffusion model *is* a control problem.

## 3. The forward process: an Ornstein–Uhlenbeck bridge to noise

Pure Brownian motion has a defect as a noising process: its variance grows without bound, so it never converges to a distribution we can sample from. Diffusion models therefore add a restoring drift, giving the **variance-preserving (VP)** forward process

$$dX_t = -\tfrac{1}{2} X_t\, dt + dW_t, \qquad X_0 \sim p_{\text{data}} .$$

This is an Ornstein–Uhlenbeck process. Its Fokker–Planck equation is $\partial_t p = \nabla\!\cdot\!\big(\tfrac{x}{2}\,p\big) + \tfrac12 \Delta p$, and the stationary density solves $\tfrac{x}{2}p + \tfrac12\nabla p = 0$, i.e. $\nabla \log p = -x$: the **standard Gaussian** $\mathcal{N}(0, I)$.

Because the equation is linear, variation of constants gives the solution in closed form,

$$X_t = e^{-t/2} X_0 + \int_0^t e^{-(t-s)/2}\, dW_s \;\sim\; \mathcal{N}\!\big(\alpha_t X_0,\; \sigma_t^2 I\big), \qquad
\alpha_t = e^{-t/2}, \quad \sigma_t^2 = 1 - e^{-t},$$

where the variance follows from the Itô isometry, $\int_0^t e^{-(t-s)}ds = 1-e^{-t}$. Note $\alpha_t^2 + \sigma_t^2 = 1$ — hence *variance preserving*, if the data is standardized.

Two consequences we will use constantly:

- **No simulation is needed to noise data.** One line, $x_t = \alpha_t x_0 + \sigma_t \varepsilon$ with $\varepsilon\sim\mathcal{N}(0,I)$, jumps directly to any noise level. This is what makes training cheap.
- **The terminal law is known.** At $T = 8$ we have $\alpha_T \approx 0.018$, so $p_T \approx \mathcal{N}(0,I)$ to within a fraction of a percent, and we can start the reverse process from pure noise.

(In the discrete-time language of DDPM, $\bar\alpha_k = \alpha_{t_k}^2$.)

> **Why a smallest time $T_{\min} > 0$.** As $t \to 0$ the target sharpens back into the data distribution, which for us is supported on a curve (§6) or on quantized pixel values (§7), and the score diverges. We therefore work on $[T_{\min}, T]$ throughout — *both* when training and when sampling. For MNIST this costs nothing: pixels are 8-bit, so their quantization step is $2/255 \approx 0.008$ on the scale $[-1,1]$, well below $\sigma_{T_{\min}} = 0.032$. Noise finer than that carries no information about the data.

In [3]:
T = 8.0        # terminal time: alpha_T = 0.018, so p_T is N(0, I) to within a fraction of a percent
T_MIN = 1e-3   # smallest noise level we ever use: sigma_{T_MIN} = 0.032 (see the remark below)

alpha = lambda t: torch.exp(-t / 2)               # signal coefficient
sigma = lambda t: torch.sqrt(-torch.expm1(-t))    # noise level sqrt(1 - e^{-t}); expm1 is stable as t -> 0
log_snr = lambda t: 2 * torch.log(alpha(t) / sigma(t))   # log signal-to-noise ratio; see section 6.3

def diffuse(x0, t, eps=None):
    """One-step forward noising: x_t = alpha_t x_0 + sigma_t eps, exact for any t."""
    eps = torch.randn_like(x0) if eps is None else eps
    shape = (-1,) + (1,) * (x0.dim() - 1)             # broadcast t over sample dimensions
    return alpha(t).reshape(shape) * x0 + sigma(t).reshape(shape) * eps

for t in [T_MIN, 0.5, 2.0, T]:
    t = torch.tensor(t)
    print(f"t = {t:6.3f}:  alpha_t = {alpha(t):.4f},  sigma_t = {sigma(t):.4f},  log-SNR = {log_snr(t):6.2f}")

t =  0.001:  alpha_t = 0.9995,  sigma_t = 0.0316,  log-SNR =   6.91
t =  0.500:  alpha_t = 0.7788,  sigma_t = 0.6273,  log-SNR =   0.43
t =  2.000:  alpha_t = 0.3679,  sigma_t = 0.9299,  log-SNR =  -1.85
t =  8.000:  alpha_t = 0.0183,  sigma_t = 0.9998,  log-SNR =  -8.00


## 4. Reversing time

We now have a process that turns data into noise. We want the opposite. Two facts make it possible: one an elementary rewriting of the Fokker–Planck equation, the other a genuine theorem about stochastic processes.

### 4.1 The probability-flow ODE

Start from the Fokker–Planck equation and use the identity $\Delta p = \nabla\!\cdot\!\big(p\, \nabla \log p\big)$, valid wherever $p > 0$:

$$\partial_t p_t \;=\; -\nabla\!\cdot\!\big(f p_t\big) + \tfrac{g^2}{2}\Delta p_t
\;=\; -\nabla\!\cdot\!\Big[\Big(\underbrace{f - \tfrac{g^2}{2}\nabla \log p_t}_{=:\,v(x,t)}\Big)\, p_t\Big].$$

The right-hand side is a **continuity equation** $\partial_t p + \nabla\cdot(v p) = 0$: the same densities $p_t$ are transported by the *deterministic* vector field $v$. So the ODE

$$\dot x(t) \;=\; f(x,t) - \tfrac{g(t)^2}{2}\,\nabla \log p_t(x)$$

has exactly the marginals of the SDE. Integrate it **backwards** from $x(T) \sim \mathcal{N}(0,I)$ and $x(0)$ is a sample of $p_{\text{data}}$.

This is worth pausing on: it is a **neural ODE** in the sense of notebook 02, whose vector field happens to be $f$ corrected by the score. All the machinery there — adaptive solvers, the adjoint method, invertibility, exact likelihoods by the instantaneous change-of-variables formula — applies verbatim.

### 4.2 Anderson's reverse-time SDE

Anderson (1982) proved the stochastic counterpart: the time-reversed process solves, with $t$ running from $T$ down to $0$,

$$dX_t = \Big[f(X_t,t) - g(t)^2\,\nabla \log p_t(X_t)\Big] dt + g(t)\, d\bar W_t,$$

where $\bar W$ is a Brownian motion in reverse time. In fact both are members of a one-parameter family (Song et al., 2021): for every $\lambda \in [0,1]$,

$$dX_t = \Big[f - \tfrac{1+\lambda^2}{2} g^2 \nabla \log p_t\Big] dt + \lambda\, g\, d\bar W_t$$

has the marginals $p_t$. $\lambda = 0$ is the probability-flow ODE, $\lambda = 1$ is Anderson's SDE. We implement the whole family in one function below. For our OU forward process ($f = -x/2$, $g = 1$) this reads

$$dX_t = \Big[-\tfrac{X_t}{2} - \tfrac{1+\lambda^2}{2}\nabla \log p_t(X_t)\Big] dt + \lambda \, d\bar W_t .$$

Everything is explicit except the **score** $\nabla \log p_t$. That is the entire learning problem.

## 5. Score matching

We need $s(x,t) = \nabla \log p_t(x)$, but we only ever see *samples* from $p_t$, never its density. Three ideas, each making the previous one practical.

### 5.1 Hyvärinen's identity (2005)

The obvious objective is useless as written, since it needs the unknown $\nabla \log p$:

$$J(\theta) = \tfrac12\, \mathbb{E}_{x \sim p}\big\|s_\theta(x) - \nabla \log p(x)\big\|^2 .$$

Expand the square and integrate the cross term by parts, using $p \nabla \log p = \nabla p$ and assuming $p\, s_\theta \to 0$ at infinity:

$$-\mathbb{E}_p\big[s_\theta \cdot \nabla \log p\big] = -\int s_\theta \cdot \nabla p \,dx = \int (\nabla\!\cdot\! s_\theta)\, p\, dx .$$

Hence, up to a constant independent of $\theta$,

$$J(\theta) = \mathbb{E}_{x\sim p}\Big[\tfrac12 \|s_\theta(x)\|^2 + \nabla\!\cdot\! s_\theta(x)\Big] + \text{const}.$$

The unknown density has **disappeared**: the objective is a plain sample average. The catch is the divergence $\nabla\cdot s_\theta = \sum_i \partial_i s_\theta^i$, which costs $d$ backward passes per training step — already hopeless at the $d = 784$ of a $28\times28$ image.

### 5.2 Tweedie's formula: the score *is* a denoiser

Our $p_t$ is not an arbitrary density: it is a Gaussian smoothing of $p_{\text{data}}$. Write $p_t(x) = \int k_{\sigma_t}(x - \alpha_t x_0)\, p_{\text{data}}(x_0)\, dx_0$ and differentiate:

$$\nabla p_t(x) = \int \left(-\frac{x - \alpha_t x_0}{\sigma_t^2}\right) k_{\sigma_t}(x - \alpha_t x_0)\, p_{\text{data}}(x_0)\, dx_0 .$$

Dividing by $p_t(x)$ turns the integral into a conditional expectation (Bayes), giving **Tweedie's formula** (Robbins 1956, Efron 2011):

$$\boxed{\;\nabla \log p_t(x) \;=\; -\frac{1}{\sigma_t^2}\,\mathbb{E}\big[\,x - \alpha_t X_0 \mid X_t = x\,\big] \;=\; -\frac{1}{\sigma_t}\,\mathbb{E}\big[\,\varepsilon \mid X_t = x \,\big].\;}$$

Equivalently, the **optimal denoiser** is an affine function of the score:

$$\hat x_0(x,t) \;:=\; \mathbb{E}[X_0 \mid X_t = x] \;=\; \frac{x + \sigma_t^2\, \nabla\log p_t(x)}{\alpha_t}.$$

*Learning to score is learning to denoise.* This is the conceptual heart of the subject.

### 5.3 Denoising score matching (Vincent, 2011)

Tweedie's formula expresses the score as a conditional expectation, and a conditional expectation is exactly what least-squares regression computes: $\mathbb{E}[Y \mid X]$ minimizes $\mathbb{E}\|h(X) - Y\|^2$ over all measurable $h$. Since we *generate* the noise $\varepsilon$ ourselves, we know the regression target. Minimize over $\theta$

$$\boxed{\;\mathcal{L}(\theta) = \mathbb{E}_{t \sim \pi}\; \mathbb{E}_{x_0 \sim p_{\text{data}}}\; \mathbb{E}_{\varepsilon \sim \mathcal{N}(0,I)} \;\big\| \varepsilon_\theta\big(\alpha_t x_0 + \sigma_t \varepsilon,\; t\big) - \varepsilon \big\|^2 \;}$$

whose minimizer is $\varepsilon_\theta^\star(x,t) = \mathbb{E}[\varepsilon \mid X_t = x] = -\sigma_t \nabla \log p_t(x)$. We then set

$$s_\theta(x,t) = -\,\frac{\varepsilon_\theta(x,t)}{\sigma_t}, \qquad \hat x_0(x,t) = \frac{x - \sigma_t\,\varepsilon_\theta(x,t)}{\alpha_t}.$$

This is the DDPM loss of Ho et al. (2020). Three lines of code, no divergence, no density, and no simulation of the forward SDE — just "add known noise, predict it back". The whole of §§6–7 is this loss plus a sampler.

> Note what has been gained over §5.1: we have *not* removed the intractability by approximation, we have moved it. The minimizer of a regression we can actually run **is** the score, exactly.

## 6. A problem we can solve by hand: the lemniscate

Before trusting a neural network, we test the theory on a problem where **the score is available in closed form**.

**Gaussian mixtures are closed under the forward process.** If $p_{\text{data}} = \sum_i w_i\, \mathcal{N}(\mu_i, \Sigma_i)$, then since $X_t = \alpha_t X_0 + \sigma_t \varepsilon$ with $\varepsilon$ independent,

$$p_t = \sum_i w_i\, \mathcal{N}\!\big(\alpha_t \mu_i,\; \alpha_t^2 \Sigma_i + \sigma_t^2 I\big).$$

The simplest instance is a single Gaussian $p_{\text{data}} = \mathcal{N}(0, s^2 I)$, for which $p_t = \mathcal{N}\big(0, (\alpha_t^2 s^2 + \sigma_t^2) I\big)$ and the score is **linear**,

$$\nabla \log p_t(x) = -\frac{x}{\alpha_t^2 s^2 + \sigma_t^2},$$

so the reverse process is itself a linear SDE, solvable in closed form. A useful sanity check, but a boring picture.

**The empirical distribution.** Take instead the other extreme, $\Sigma_i = 0$: the data itself, $p_{\text{data}} = \frac1N \sum_i \delta_{x_i}$. Then $p_t$ is a mixture of $N$ isotropic Gaussians and

$$\nabla \log p_t(x) = -\frac{1}{\sigma_t^2} \sum_{i=1}^N w_i(x,t)\,\big(x - \alpha_t x_i\big),
\qquad w_i(x,t) = \operatorname{softmax}_i\!\left(-\frac{\|x - \alpha_t x_i\|^2}{2\sigma_t^2}\right).$$

Tweedie's formula then gives the denoiser a very concrete meaning, $\hat x_0(x,t) = \sum_i w_i(x,t)\, x_i$ — a **soft nearest neighbour** among the training points, at temperature $\sigma_t^2$. Early in the reverse process ($\sigma_t$ large) the weights are nearly uniform and $\hat x_0$ is the data mean; late ($\sigma_t \to 0$) they concentrate on a single training point. Keep that last observation in mind: we will watch it happen.

For the data we use the **lemniscate of Bernoulli**, the figure-eight $\big(x_1^2 + x_2^2\big)^2 = a^2\big(x_1^2 - x_2^2\big)$, sampled through the rational parametrization

$$x_1(\tau) = \frac{a\cos\tau}{1 + \sin^2\tau}, \qquad x_2(\tau) = \frac{a\sin\tau\cos\tau}{1 + \sin^2\tau}, \qquad \tau \in [0, 2\pi).$$

It is a good test case: a one-dimensional set embedded in the plane, so the target is singular with respect to Lebesgue measure, with a self-intersection at the origin and two lobes that the reverse process has to keep apart.

In [ ]:
def lemniscate(n, a=1.6):
    """Sample n points on Bernoulli's lemniscate (x1^2+x2^2)^2 = a^2 (x1^2-x2^2)."""
    tau = 2 * np.pi * torch.rand(n)
    d = 1 + torch.sin(tau) ** 2
    return torch.stack([a * torch.cos(tau) / d, a * torch.sin(tau) * torch.cos(tau) / d], dim=1)

X_data = lemniscate(128)            # the training set: deliberately small, so we can see what it does
X_curve = lemniscate(20000)         # a dense reference for "the lemniscate itself"

spacing = torch.cdist(X_data, X_data).fill_diagonal_(np.inf).min(dim=1).values.mean()
print(f"{len(X_data)} training points, mean nearest-neighbour spacing = {spacing:.4f}")

# Forward process: the data dissolves into N(0, I)
fig, axes = panels(5, size=(2.6, 2.8), sharex=True, sharey=True)
for ax, t in zip(axes, [T_MIN, 0.05, 0.25, 1.0, T]):
    xt = diffuse(X_data, torch.full((len(X_data),), t))
    ax.plot(xt[:, 0], xt[:, 1], ".", ms=4, color=BLUE)
    label(ax, title=f"$t = {t}$   ($\\sigma_t = {sigma(torch.tensor(t)):.2f}$)",
          xlim=(-3, 3), ylim=(-3, 3), equal=True)
fig.suptitle("Forward process: $x_t = \\alpha_t x_0 + \\sigma_t \\varepsilon$", y=1.04)
plt.show()

# The closed-form kernel really is the law of the SDE: check against Euler-Maruyama
x0, h, n_paths = torch.tensor([[1.5, 0.3]]), 0.001, 4000
x = x0.repeat(n_paths, 1)
for _ in range(1000):                                   # simulate dX = -X/2 dt + dW up to t = 1
    x = x - 0.5 * x * h + np.sqrt(h) * torch.randn_like(x)
t1 = torch.tensor(1.0)
fmt = lambda v: "[" + ", ".join(f"{c:.3f}" for c in v) + "]"
print(f"Euler-Maruyama at t = 1:  mean = {fmt(x.mean(0))},  std = {x.std(0).mean():.4f}")
print(f"closed form            :  mean = {fmt(alpha(t1) * x0[0])},  std = {sigma(t1):.4f}")

### 6.1 Sampling with the exact score

We integrate the reverse-time family of §4.2 with an Euler–Maruyama step. Three numerical points deserve attention:

- **The time grid is geometric, not uniform.** Near $t = 0$ the score of an empirical measure behaves like $\|x - \alpha_t x_i\|/\sigma_t^2$, so the reverse dynamics are **stiff**: an explicit step of size $h$ is stable only while $h \lesssim \sigma_t^2 \approx t$. A grid with $t_{k+1}/t_k$ constant gives exactly $h_k \propto t_k \propto \sigma_{t_k}^2$, keeping the effective step size constant throughout.
- **We stop at $T_{\min}$ and finish with one denoising step,** replacing $x_{T_{\min}}$ by $\hat x_0 = \mathbb{E}[X_0 \mid X_{T_{\min}}]$. This removes the last $\mathcal{O}(\sigma_{T_{\min}})$ of noise for free, using nothing but Tweedie's formula.
- **The sampler must stay inside the range the score is valid on.** Irrelevant here, where `exact_score` is exact for every $t>0$ — but in §6.3 it becomes the difference between a clean figure-eight and a smear, so we fix the convention now: integrate on $[T_{\min}, T]$, train on $[T_{\min}, T]$.

In [5]:
@torch.no_grad()
def reverse_sample(score, x, ts, lam=1.0):
    """Integrate dx = [-x/2 - (1+lam^2)/2 * score] dt + lam dW backwards along the decreasing grid ts.

    lam = 0 -> probability-flow ODE,  lam = 1 -> Anderson's reverse SDE.
    """
    traj = [x]
    for i in range(len(ts) - 1):
        t, h = ts[i], ts[i] - ts[i + 1]
        x = x + h * (0.5 * x + 0.5 * (1 + lam ** 2) * score(x, t))
        if lam > 0 and i < len(ts) - 2:                  # no noise on the very last step
            x = x + lam * h.sqrt() * torch.randn_like(x)
        traj.append(x)
    return torch.stack(traj)

def time_grid(n, t_min=T_MIN):
    """Geometric grid T = t_0 > ... > t_{n-1} = t_min, so that h_k is proportional to sigma_{t_k}^2."""
    return torch.logspace(np.log10(T), np.log10(t_min), n)

@torch.no_grad()
def denoise(score, x, t):
    """Tweedie's formula: the posterior mean E[x_0 | x_t = x]."""
    return (x + sigma(t) ** 2 * score(x, t)) / alpha(t)

def exact_score(x, t):
    """Exact score of p_t when p_data is the empirical measure of X_data (a Gaussian mixture)."""
    d = x[:, None, :] - alpha(t) * X_data[None, :, :]                  # (B, N, 2)
    w = torch.softmax(-(d ** 2).sum(-1) / (2 * sigma(t) ** 2), dim=1)  # soft nearest neighbour
    return -(w[:, :, None] * d).sum(1) / sigma(t) ** 2

In [ ]:
ts_2d = time_grid(400)
torch.manual_seed(1)
x_T = torch.randn(1024, 2)                                    # the same noise for both samplers

traj_sde = reverse_sample(exact_score, x_T, ts_2d, lam=1.0)   # Anderson's reverse SDE
traj_ode = reverse_sample(exact_score, x_T, ts_2d, lam=0.0)   # probability-flow ODE
gen_sde = denoise(exact_score, traj_sde[-1], ts_2d[-1])       # final Tweedie step
gen_ode = denoise(exact_score, traj_ode[-1], ts_2d[-1])

fig, axes = panels(3, size=(3.7, 3.4), sharex=True, sharey=True)
for ax, (pts, ttl, col) in zip(axes, [(X_data, "data $p_0$ (128 points)", GREY),
                                      (gen_sde, r"reverse SDE ($\lambda = 1$)", BLUE),
                                      (gen_ode, r"probability-flow ODE ($\lambda = 0$)", ORANGE)]):
    ax.plot(X_curve[:, 0], X_curve[:, 1], ".", ms=0.4, color=GHOST)
    ax.plot(pts[:, 0], pts[:, 1], ".", ms=3.5, color=col)
    label(ax, title=ttl, equal=True, xlim=(-2.1, 2.1), ylim=(-1.1, 1.1))
plt.show()

Both samplers reproduce the figure-eight from pure noise. Look closely, though: the generated points are not spread along the curve, they sit in **clumps**. The clumps are the 128 training points.

That is not a bug, it is the exact answer. We handed the sampler the true score of $\frac{1}{N}\sum_i \delta_{x_i}$, and the true score of the empirical measure generates the empirical measure — as the soft-nearest-neighbour form of $\hat x_0$ already told us it would. **A model that fits the data distribution perfectly memorizes it.**

We can measure this. For each generated point, record the distance to the nearest *training point* and the distance to the *curve* (a dense reference sample), and compare both against two yardsticks: the spacing between neighbouring training points, and the value a fresh sample of the true lemniscate would achieve.

In [7]:
def report(name, gen):
    d_train = torch.cdist(gen, X_data).min(dim=1).values.mean()
    d_curve = torch.cdist(gen, X_curve).min(dim=1).values.mean()
    print(f"{name:28s}  to nearest training point {d_train:.4f}   to the lemniscate {d_curve:.4f}")

report("reverse SDE, exact score", gen_sde)
report("prob.-flow ODE, exact score", gen_ode)
report("fresh lemniscate samples", lemniscate(1024))
print(f"\nfor reference, the mean spacing between training points is {spacing:.4f}")

reverse SDE, exact score      to nearest training point 0.0080   to the lemniscate 0.0007
prob.-flow ODE, exact score   to nearest training point 0.0084   to the lemniscate 0.0007
fresh lemniscate samples      to nearest training point 0.0299   to the lemniscate 0.0002

for reference, the mean spacing between training points is 0.0329


The exact score puts samples on the curve (`d_curve` a few thousandths) and, crucially, **at a quarter of the inter-point spacing from a training point** — much closer than a fresh draw from the true lemniscate manages. It reproduces the training set, not the curve the training set was drawn from.

This is the sharpest available statement of what a generative model actually needs. Not a better estimate of $p_{\text{data}}$ — the best estimate is useless. What is needed is an **inductive bias**: a score class smooth enough that it cannot represent the spikes of an empirical measure, and interpolates between training points instead. That is what §6.3 demonstrates.

Note also the difference between the two panels above. The ODE ($\lambda = 0$) is a deterministic bijection between noise and data, so 1024 Gaussian samples map to 1024 outputs that inherit the geometry of the noise; the SDE ($\lambda = 1$) keeps injecting randomness and mixes between neighbouring basins. Same marginals in theory, different sensitivity to discretization error in practice.

### 6.2 The reverse process in motion

The animation follows all 1024 samples as $t$ decreases from $T$ to $T_{\min}$. Watch for two phases, with the transition where $\sigma_t$ crosses the scale of the data: the Gaussian blob first contracts and commits to a lobe (low-frequency information, decided early), and only then collapses onto the curve (fine detail, decided late).

In [ ]:
frames = torch.linspace(0, len(traj_sde) - 1, 40).long()

animate_points(traj_sde[frames], background=X_curve, s=4, figsize=(4.2, 3.7),
               xlabel="$x_1$", ylabel="$x_2$", xlim=(-3.2, 3.2), ylim=(-3.2, 3.2),
               title=lambda k: f"reverse SDE at $t = {ts_2d[frames[k]]:.3f}$   "
                               f"($\\sigma_t = {sigma(ts_2d[frames[k]]):.3f}$)")

### 6.3 Learning the score instead of computing it

Now we forget the closed form and estimate the score from the 128 data points by denoising score matching. The network is a small MLP taking the two coordinates of $x$ together with the noise level.

**How to tell the network the time.** Not by handing it $t$. What the network needs to know is how much signal survives, measured by the **log signal-to-noise ratio**

$$\lambda_t \;=\; \log \frac{\alpha_t^2}{\sigma_t^2} \;=\; -t - \log\big(1 - e^{-t}\big),$$

which decreases from $\lambda_{T_{\min}} = 6.9$ (almost clean) to $\lambda_T = -8.0$ (pure noise). This is the natural coordinate on the schedule: equal increments of $\lambda$ are equally large changes in the denoising problem. The obvious alternative, $\log\sigma_t$, is a poor coordinate: it compresses the whole of $t \in [1, 8]$ into an interval of width $0.23$, leaving the network almost no resolution in precisely the regime where an image's global layout is decided. We feed $\lambda_t/8$ through a small MLP; the same `TimeEmbedding` is reused unchanged for the image model in §7.

**Which times to train on.** The loss averages over $t \sim \pi$, and $\pi$ is a design choice. Uniform on $[0,T]$ is wasteful: with $\sigma_t^2 = 1 - e^{-t}$, more than half of $[0,8]$ has $\sigma_t > 0.98$, where the score is nearly the trivial $-x$ and there is nothing to learn. We sample $t$ **log-uniformly** on $[T_{\min}, T]$, which matches the geometric grid used at sampling time and spreads the budget evenly over orders of magnitude of $\sigma_t$.

That match is not cosmetic. A network trained on $[10^{-2}, T]$ but integrated down to $10^{-3}$ is asked to extrapolate exactly where the dynamics are stiffest, and the figure-eight comes out as a blurred cloud — a mistake worth making once, deliberately (Exercise 3).

In [9]:
class TimeEmbedding(nn.Module):
    """Turn the noise level into a vector: the log-SNR, rescaled, through a small MLP."""
    def __init__(self, dim=32):
        super().__init__()
        self.dim = dim
        self.net = nn.Sequential(nn.Linear(1, dim), nn.SiLU(), nn.Linear(dim, dim))
    def forward(self, t, n):
        return self.net((log_snr(t) / 8).reshape(-1, 1).expand(n, 1))      # scalar t broadcasts

class ScoreNet(nn.Module):
    """Predicts the noise eps given (x, t); the score is then -eps_theta / sigma_t."""
    def __init__(self, dim=2, width=128):
        super().__init__()
        self.temb = TimeEmbedding()
        self.net = nn.Sequential(
            nn.Linear(dim + self.temb.dim, width), nn.SiLU(),
            nn.Linear(width, width), nn.SiLU(),
            nn.Linear(width, width), nn.SiLU(),
            nn.Linear(width, dim),
        )
    def forward(self, x, t):
        return self.net(torch.cat([x, self.temb(t, len(x))], dim=1))

def sample_times(n):
    """Log-uniform times on [T_MIN, T] -- the same range the sampler will integrate over."""
    return T_MIN * (T / T_MIN) ** torch.rand(n)

In [10]:
torch.manual_seed(0)
eps_theta = ScoreNet()
opt = torch.optim.Adam(eps_theta.parameters(), lr=3e-3)
STEPS = 6000
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS)

losses = []
for it in range(STEPS + 1):
    x0 = X_data[torch.randint(0, len(X_data), (256,))]      # a batch of data
    t = sample_times(256)                                   # a batch of noise levels
    eps = torch.randn_like(x0)                              # the regression target
    loss = ((eps_theta(diffuse(x0, t, eps), t) - eps) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    losses.append(loss.item())
    if it % 1500 == 0:
        print(f"iteration {it:5d}:  loss = {np.mean(losses[-200:]):.4f}")

learned_score = lambda x, t: -eps_theta(x, t) / sigma(t)

iteration     0:  loss = 1.0463


iteration  1500:  loss = 0.5937


iteration  3000:  loss = 0.4492


iteration  4500:  loss = 0.4229


iteration  6000:  loss = 0.4186


In [11]:
# How close is the learned score to the exact one, measured on samples of p_t?
print("relative L2 error of the learned score, on samples of p_t:")
with torch.no_grad():
    for t in [T_MIN, 0.05, 0.5, 2.0]:
        t = torch.tensor(t)
        xs = diffuse(X_data, t.repeat(len(X_data)))
        s_exact, s_hat = exact_score(xs, t), learned_score(xs, t)
        print(f"   t = {t:6.3f}  (sigma_t = {sigma(t):.3f}):  "
              f"{(s_hat - s_exact).norm() / s_exact.norm():.3f}")

relative L2 error of the learned score, on samples of p_t:
   t =  0.001  (sigma_t = 0.032):  0.619
   t =  0.050  (sigma_t = 0.221):  0.163
   t =  0.500  (sigma_t = 0.627):  0.035
   t =  2.000  (sigma_t = 0.930):  0.044


In [ ]:
# The two score fields side by side
t_show = torch.tensor(0.3)
g = torch.linspace(-2.6, 2.6, 22)
G = torch.stack(torch.meshgrid(g, g, indexing="xy"), dim=-1).reshape(-1, 2)
with torch.no_grad():
    fields = [exact_score(G, t_show), learned_score(G, t_show)]

fig, axes = panels(2, size=(4.5, 4.0))
for ax, s, ttl in zip(axes, fields, [f"exact score at $t = {t_show:.1f}$", "learned score, same $t$"]):
    u = s / (1 + s.norm(dim=1, keepdim=True))                     # soft normalization, for legibility
    ax.quiver(G[:, 0], G[:, 1], u[:, 0], u[:, 1], s.norm(dim=1), cmap="viridis", scale=22, width=0.004)
    ax.plot(*(alpha(t_show) * X_data).T, ".", ms=4, color=MAGENTA)
    label(ax, title=ttl, equal=True)
plt.show()

In [ ]:
# What the learned score generates, and how far it stays from the training points
with torch.no_grad():
    gen_learned = denoise(learned_score, reverse_sample(learned_score, x_T, ts_2d, lam=1.0)[-1], ts_2d[-1])

report("reverse SDE, exact score", gen_sde)
report("reverse SDE, learned score", gen_learned)
report("fresh lemniscate samples", lemniscate(1024))

fig, axes = panels(2, size=(5.25, 3.6))
axes[0].plot(X_curve[:, 0], X_curve[:, 1], ".", ms=0.4, color=GHOST)
axes[0].plot(X_data[:, 0], X_data[:, 1], ".", ms=7, color=GREY, label="training points")
axes[0].plot(gen_learned[:, 0], gen_learned[:, 1], ".", ms=2.5, color=BLUE, label="generated")
label(axes[0], title="samples from the learned score", equal=True,
      xlim=(-2.1, 2.1), ylim=(-1.1, 1.1), legend={"loc": "upper right"})

bins = np.linspace(0, 0.09, 40)
axes[1].hist(torch.cdist(gen_sde, X_data).min(1).values.numpy(), bins=bins, alpha=0.75,
             color=ORANGE, label="exact score")
axes[1].hist(torch.cdist(gen_learned, X_data).min(1).values.numpy(), bins=bins, alpha=0.75,
             color=BLUE, label="learned score")
axes[1].axvline(spacing, color=INK, ls="--", lw=1.2)
axes[1].text(spacing * 1.05, axes[1].get_ylim()[1] * 0.9, "training-point\nspacing", fontsize=8)
label(axes[1], "distance to the nearest training point", "count")
plt.show()

The two vector fields are nearly indistinguishable away from the data, and the relative error printed above is a few percent at moderate noise, growing as $t \to T_{\min}$ where the exact score becomes singular and the network stays smooth.

That failure to reproduce the singularity is the entire point. The exact score piles its samples up at distances well below the training-point spacing; the learned score produces a histogram that looks like the one a **fresh draw from the true lemniscate** would produce, while `d_curve` stays small. The finite-capacity network has turned an empirical measure of 128 atoms into a distribution on the underlying curve — it **generalizes**, precisely by failing to solve its stated optimization problem exactly.

The same smoothness has a price, visible as the small pile-up of samples at the origin. The node of the lemniscate is the one point where the curve is not a manifold: two branches cross, the score has no continuous extension there, and a smooth network resolves the conflict by averaging. Generalization and artefacts at singularities are two readings of one fact.

## 7. A small diffusion model on MNIST

Nothing changes except the dimension: $d = 2$ becomes $d = 784$. The loss is the same, the sampler is *literally the same function*; only the network has to be adapted to the fact that the data are images.

**Why a convolutional U-Net.** The score $\nabla \log p_t(x)$ is a vector field on image space — one number per pixel. Two structural facts guide the architecture: the score is approximately *local* (a pixel's noise is best estimated from its neighbourhood), and it acts at *all scales* (at large $t$ only the coarse layout survives, at small $t$ only fine texture is left to fix). A U-Net gives both: convolutions for locality, downsampling plus skip connections for scale.

**Time conditioning.** Exactly the same `TimeEmbedding` as in the plane, but its output vector is now projected into the channel dimension and **added to every block**, the way a positional encoding is added in a transformer. Every layer, at every scale, needs to know the noise level.

**Standardize the data.** The forward process has $\mathcal{N}(0,I)$ as its stationary law, and we start the reverse process from exactly that — so $p_{\text{data}}$ has to meet it. MNIST scaled to $[-1,1]$ has mean $-0.74$ and standard deviation $0.61$, and the mismatch is not cosmetic. The slowest mode for the forward process to destroy is the image-wide average brightness: its signal decays as $\alpha_T |\bar x|$ while the noise in that one mode is only $\sigma_T/\sqrt{d}$, leaving a residual signal-to-noise ratio of $|\bar x|\,\alpha_T \sqrt{d}/\sigma_T \approx 0.38$ at $t = T$. The forward process has *not* forgotten the brightness. Starting the reverse process from a genuinely centred Gaussian then hands the network a global grey level it never saw in training, and the digits come out with plausible strokes on a wrong background. Subtracting the mean and dividing by the standard deviation removes the problem at its root, and the cell below prints the residual dropping from $0.38$ to $0.07$. (This is the issue analysed by Lin et al., 2024; the other standard remedy is to push $T$ out until $\alpha_T$ is negligible.)

**Why predict $\varepsilon$ and not the score.** The score has magnitude $\sim 1/\sigma_t$, which ranges over two orders of magnitude on $[T_{\min}, T]$; the noise $\varepsilon$ is $\mathcal{N}(0,I)$ at every $t$. Predicting $\varepsilon$ gives the network an $\mathcal{O}(1)$ target uniformly in $t$ and implicitly weights the loss in the way that works well in practice.

In [ ]:
# MNIST: 60000 handwritten digits, 28 x 28, downloaded once into ./data
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
gz = DATA / "mnist-train-images.gz"
if not gz.exists():
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz", gz)

raw = np.frombuffer(gzip.decompress(gz.read_bytes()), dtype=np.uint8, offset=16)
mnist = torch.tensor(raw.reshape(-1, 1, 28, 28).copy())          # uint8, converted per batch
print("dataset:", tuple(mnist.shape))

# Standardize, so that p_data really does have the mean and variance of the stationary law N(0, I)
_pixels = mnist[:10000].float() / 127.5 - 1
MU, SD = _pixels.mean(), _pixels.std()
to_model = lambda u8: ((u8.float() / 127.5 - 1) - MU) / SD       # uint8 image  -> model coordinates
to_image = lambda x: x * SD + MU                                 # model coords -> [-1, 1] for display
print(f"on [-1,1] the pixels have mean {MU:.3f} and std {SD:.3f}; the model sees them standardized")

# How much image-wide brightness still survives at t = T? Signal alpha_T |DC|, noise sigma_T / sqrt(d).
tT, d = torch.tensor(T), 28 * 28
for name, x in [("raw, on [-1,1] ", mnist[:10000].float() / 127.5 - 1), ("standardized", to_model(mnist[:10000]))]:
    dc = x.mean(dim=(1, 2, 3))                                   # the spatial mean of each image
    print(f"   residual brightness SNR at t = T ({name}): "
          f"{dc.pow(2).mean().sqrt() * alpha(tT) * np.sqrt(d) / sigma(tT):.3f}")

show_images(mnist[:16].float() / 127.5 - 1, nrow=8, title="training data", figsize=(8, 1.2))
plt.show()

In [15]:
class Block(nn.Module):
    """Two convolutions with group norm, plus an additive time embedding."""
    def __init__(self, c_in, c_out, c_emb):
        super().__init__()
        self.conv1, self.conv2 = nn.Conv2d(c_in, c_out, 3, padding=1), nn.Conv2d(c_out, c_out, 3, padding=1)
        self.norm1, self.norm2 = nn.GroupNorm(8, c_out), nn.GroupNorm(8, c_out)
        self.emb = nn.Linear(c_emb, c_out)
    def forward(self, x, e):
        h = F.silu(self.norm1(self.conv1(x))) + self.emb(e)[:, :, None, None]
        return F.silu(self.norm2(self.conv2(h)))

class UNet(nn.Module):
    """Minimal three-scale U-Net: 28 -> 14 -> 7 -> 14 -> 28."""
    def __init__(self, c=16, c_emb=64):
        super().__init__()
        self.temb = TimeEmbedding(c_emb)                              # same module as in section 6
        self.down1, self.down2 = Block(1, c, c_emb), Block(c, 2 * c, c_emb)
        self.mid = Block(2 * c, 2 * c, c_emb)
        self.up2, self.up1 = Block(4 * c, c, c_emb), Block(2 * c, c, c_emb)
        self.out = nn.Conv2d(c, 1, 1)

    def forward(self, x, t):
        e = self.temb(t, len(x))
        h1 = self.down1(x, e)                                         # 28 x 28
        h2 = self.down2(F.avg_pool2d(h1, 2), e)                       # 14 x 14
        m = self.mid(F.avg_pool2d(h2, 2), e)                          #  7 x  7
        u = self.up2(torch.cat([F.interpolate(m, scale_factor=2), h2], dim=1), e)
        u = self.up1(torch.cat([F.interpolate(u, scale_factor=2), h1], dim=1), e)
        return self.out(u)

unet = UNet()
print(f"parameters: {sum(p.numel() for p in unet.parameters()):,}")

parameters: 65,393


The training loop is the same four lines as in the plane: draw images, draw times, add noise, predict it back. Nothing about it knows that the samples are pictures.

> This cell is the only slow one in the notebook — roughly ten minutes on a laptop CPU. The model is deliberately tiny (65k parameters, three scales); the digits it produces are legible rather than beautiful, and both quality and training time scale with `c`, `BATCH` and `STEPS`.

In [16]:
STEPS, BATCH = 6000, 64
opt = torch.optim.Adam(unet.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS)

mnist_losses = []
for it in range(STEPS + 1):
    x0 = to_model(mnist[torch.randint(0, len(mnist), (BATCH,))])            # standardized batch
    t = sample_times(BATCH)
    eps = torch.randn_like(x0)
    loss = ((unet(diffuse(x0, t, eps), t) - eps) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    mnist_losses.append(loss.item())
    if it % 1000 == 0:
        print(f"iteration {it:5d}:  loss = {np.mean(mnist_losses[-200:]):.4f}")

iteration     0:  loss = 1.1050


iteration  1000:  loss = 0.1009


iteration  2000:  loss = 0.0903


iteration  3000:  loss = 0.0874


iteration  4000:  loss = 0.0846


iteration  5000:  loss = 0.0848


iteration  6000:  loss = 0.0829


In [ ]:
mnist_score = lambda x, t: -unet(x, t) / sigma(t)

ts_img = time_grid(300)
torch.manual_seed(3)
traj_img = reverse_sample(mnist_score, torch.randn(16, 1, 28, 28), ts_img, lam=1.0)
digits = to_image(denoise(mnist_score, traj_img[-1], ts_img[-1])).clamp(-1, 1)

show_images(digits, nrow=8, figsize=(8, 2.4),
            title="samples from the reverse SDE, started from pure noise")
plt.show()

### 7.1 Watching the denoising

The animation shows both halves of Tweedie's formula along one reverse trajectory:

- **top**: the state $x_t$ of the reverse SDE — noise that slowly organizes itself;
- **bottom**: the network's current guess $\hat x_0(x_t,t) = \big(x_t - \sigma_t \varepsilon_\theta(x_t,t)\big)/\alpha_t$ of where that trajectory will end.

Three things are worth watching for.

**The first frame is noise, and necessarily so.** At $t = T$ the denoiser divides by $\alpha_T = 0.018$, amplifying whatever error $\varepsilon_\theta$ makes by a factor of $55$. The right answer there is almost the dataset average, but no network is accurate enough to survive that gain, so the bottom row starts as salt and pepper. This is the practical argument for predicting $\varepsilon$ rather than $x_0$: the quantity we regress on stays $\mathcal{O}(1)$ at every noise level, and the unstable division happens only where we choose to look.

**Identity is settled long before detail.** By $t \approx 1.4$, where $\alpha_t \approx 0.5$, the bottom row already shows *which* digit each trajectory will become, while the top row still looks like noise. The remaining two thirds of the trajectory only fix strokes and thickness.

**Generation is coarse-to-fine**, which is §1 read backwards: the heat equation destroys high frequencies first, so reversing it restores them last.

In [ ]:
idx = torch.linspace(0, len(traj_img) - 1, 32).long()             # 32 frames, 6 digits each
frames_x = to_image(traj_img[idx]).clamp(-1, 1)
frames_x0 = to_image(torch.stack([denoise(mnist_score, traj_img[k], ts_img[k]) for k in idx])).clamp(-1, 1)
film = [np.vstack([image_grid(a[:6], 6), image_grid(b[:6], 6)]) for a, b in zip(frames_x, frames_x0)]

animate_images(film, title=lambda k: f"$t = {ts_img[idx[k]]:.3f}$,   "
                                     f"$\\sigma_t = {sigma(ts_img[idx[k]]):.3f}$\n"
                                     "top: $x_t$    bottom: $\\hat{x}_0(x_t,t)$")

## 8. Outlook: diffusion as stochastic control

The reverse process is a *controlled* SDE, and this is not a metaphor. Consider the problem of steering the process

$$dX_t = u(X_t,t)\,dt + dW_t, \qquad X_0 \sim \mathcal{N}(0,I),$$

so that $X_T \sim p_{\text{data}}$ at minimal control energy $\tfrac12\mathbb{E}\int_0^T \|u\|^2 dt$. This is the **Schrödinger bridge** problem (Schrödinger, 1931): find the law on path space closest in relative entropy to the Brownian reference measure, with prescribed marginals at both ends. Its solution is characterized by a Hamilton–Jacobi–Bellman equation which, under the Cole–Hopf transform $u = \nabla \log \varphi$, linearizes into exactly the Feynman–Kac / heat equation pair of §2 — a system of forward and backward heat equations coupled through the boundary conditions.

A diffusion model is the case in which the reference process is chosen so that one of the two constraints holds automatically: the OU process of §3 already carries $p_{\text{data}}$ to $\mathcal{N}(0,I)$ to within $\alpha_T = 0.018$. No iteration is then needed, and the optimal control is available in closed form — it is the score. The general bridge requires iterating (Sinkhorn / iterative proportional fitting), and is an active research area — De Bortoli et al. (2021), and the closely related **flow matching** family (Lipman et al., 2023), which learns the probability-flow vector field of §4.1 directly by regression, and connects back to the neural ODEs of notebook 02.

Three directions worth exploring from here:

- **Conditional generation.** Train $\varepsilon_\theta(x,t,y)$ with a label $y$, and note that Bayes gives $\nabla\log p_t(x\mid y) = \nabla\log p_t(x) + \nabla \log p_t(y \mid x)$: guidance is a score decomposition, not a hack.
- **Inverse problems.** Replace the label by a measurement operator and the same decomposition turns a pretrained diffusion model into a prior for deblurring, inpainting or tomography.
- **Sampler design.** Everything in §4.2 was one family of SDEs with identical marginals but different discretization errors. Choosing $\lambda$, the time grid and the integrator is where most of the practical speed-up of the last years came from (Karras et al., 2022).

### Exercises

1. Show that the single-Gaussian case of §6 gives a **linear** reverse SDE, solve it explicitly, and verify numerically against `reverse_sample`.
2. Verify Hyvärinen's identity numerically in 1D: estimate $\mathbb{E}_p[\tfrac12\|s_\theta\|^2 + \partial_x s_\theta]$ by Monte Carlo for a Gaussian $p$ and compare with $J(\theta)$ computed in closed form.
3. Replace the geometric time grid by a uniform one at the same number of steps, and find where the 2-D sampler becomes unstable. Relate the threshold to $h \lesssim \sigma_t^2$.
4. Compare $\lambda = 0$ and $\lambda = 1$ on MNIST at 30, 100 and 300 steps. Which degrades faster, and why?
5. Add class conditioning to the MNIST model (the labels are in `train-labels-idx1-ubyte.gz`) and implement classifier-free guidance by training with the label randomly dropped.

### References

**The classical background**

- J. Fourier, *Théorie analytique de la chaleur*, 1822 — the heat equation.
- A. Einstein, "Über die von der molekularkinetischen Theorie der Wärme geforderte Bewegung...", *Ann. Phys.* 322 (1905); M. Smoluchowski (1906) — Brownian motion as a diffusion. N. Wiener (1923) — its rigorous construction.
- A. N. Kolmogorov, "Über die analytischen Methoden in der Wahrscheinlichkeitsrechnung", *Math. Ann.* 104 (1931) — the forward and backward equations.
- R. P. Feynman, "Space-time approach to non-relativistic quantum mechanics", *Rev. Mod. Phys.* 20 (1948); M. Kac, "On distributions of certain Wiener functionals", *Trans. AMS* 65 (1949) — the Feynman–Kac formula.
- E. Schrödinger, "Über die Umkehrung der Naturgesetze", *Sitzungsberichte Preuss. Akad. Wiss.* (1931) — the bridge problem; H. Föllmer, "Random fields and diffusion processes" (1988).
- B. D. O. Anderson, "Reverse-time diffusion equation models", *Stochastic Processes and their Applications* 12 (1982) — the time-reversal formula that §4.2 rests on.
- H. Robbins (1956); B. Efron, "Tweedie's formula and selection bias", *JASA* 106 (2011) — the denoiser/score identity.

**Learning the score**

- A. Hyvärinen, "Estimation of non-normalized statistical models by score matching", *JMLR* 6 (2005).
- P. Vincent, "A connection between score matching and denoising autoencoders", *Neural Computation* 23 (2011).

**Diffusion models**

- J. Sohl-Dickstein, E. Weiss, N. Maheswaranathan, S. Ganguli, "Deep unsupervised learning using nonequilibrium thermodynamics", *ICML* 2015 — the first diffusion model.
- Y. Song, S. Ermon, "Generative modeling by estimating gradients of the data distribution", *NeurIPS* 2019 — noise-conditional score networks.
- J. Ho, A. Jain, P. Abbeel, "Denoising diffusion probabilistic models", *NeurIPS* 2020 — the $\varepsilon$-prediction loss used here.
- Y. Song, J. Sohl-Dickstein, D. Kingma, A. Kumar, S. Ermon, B. Poole, "Score-based generative modeling through stochastic differential equations", *ICLR* 2021 — the continuous-time picture, the probability-flow ODE, the $\lambda$-family.
- T. Karras, M. Aittala, T. Aila, S. Laine, "Elucidating the design space of diffusion-based generative models", *NeurIPS* 2022 — schedules, preconditioning, samplers.
- V. De Bortoli, J. Thornton, J. Heng, A. Doucet, "Diffusion Schrödinger bridge", *NeurIPS* 2021; Y. Lipman, R. Chen, H. Ben-Hamu, M. Nickel, M. Le, "Flow matching for generative modeling", *ICLR* 2023.
- S. Lin, B. Liu, J. Li, X. Yang, "Common diffusion noise schedules and sample steps are flawed", *WACV* 2024 — why the residual signal at $t = T$ matters, as in §7.
- R. T. Q. Chen, Y. Rubanova, J. Bettencourt, D. Duvenaud, "Neural ordinary differential equations", *NeurIPS* 2018 — notebook 02, and the ODE of §4.1.